In [1]:
import sqlite3
import csv
import urllib.request
import os

#Github raw URLs for the three CSV files

BASE_URL = "https://raw.githubusercontent.com/JahleekMitchell/CIS3120/JahleekMitchell-module07/"

BOOK_URL = BASE_URL + "Book.csv"
MEMBER_URL = BASE_URL + "Member.csv"
LOAN_URL = BASE_URL + "Loan.csv"

#Local paths in the content directory
BOOK_PATH = "Book.csv"
MEMBER_PATH = "Member.csv"
LOAN_PATH = "Loan.csv"

DB_PATH = "library.db"

In [2]:
for url, path in [(BOOK_URL, BOOK_PATH), (MEMBER_URL, MEMBER_PATH), (LOAN_URL, LOAN_PATH)]:
    urllib.request.urlretrieve(url, path)
    print(f"Downloaded: {path}")

Downloaded: Book.csv
Downloaded: Member.csv
Downloaded: Loan.csv


In [3]:
conn = sqlite3.connect(DB_PATH)

conn.execute('PRAGMA foreign_keys = ON;')

 

# CREATE TABLE statements go here

# Book Table
conn.execute('''
CREATE TABLE IF NOT EXISTS Book (
    callNo  TEXT    NOT NULL,
    title   TEXT    NOT NULL,
    author  TEXT    NOT NULL,
    PRIMARY KEY (callNo)
);
''')

# Member Table
conn.execute('''
CREATE TABLE IF NOT EXISTS Member (
    id          INTEGER NOT NULL,
    firstname   TEXT    NOT NULL,
    lastName    TEXT    NOT NULL,
    PRIMARY KEY (id)
);
''')
 
# Loan table
conn.execute('''
CREATE TABLE IF NOT EXISTS Loan (
    callNo        TEXT    NOT NULL,
    id            INTEGER NOT NULL,
    dateBorrowed  TEXT    NOT NULL,
    dateReturned  TEXT,
    dateDue       TEXT    NOT NULL,
    PRIMARY KEY (callNo, id, dateBorrowed),
    FOREIGN KEY (callNo) REFERENCES Book(callNo),
    FOREIGN KEY (id)     REFERENCES Member(id)
);
''')
conn.commit()

print("Tables created.")

Tables created.


In [4]:
conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;").fetchall()

[('Book',), ('Loan',), ('Member',)]

In [5]:
with open(BOOK_PATH, newline='', encoding='utf-8') as f:

    reader = csv.DictReader(f)

    for row in reader:

        conn.execute(

            'INSERT INTO Book (callNo, title, author) VALUES (?, ?, ?);',

            (row['callNo'], row['title'], row['author'])

        )

 

conn.commit()

print('Book rows loaded:', conn.execute('SELECT COUNT(*) FROM Book;').fetchone()[0])

Book rows loaded: 11


In [6]:
with open(MEMBER_PATH, newline='', encoding='utf-8') as f:

    reader = csv.DictReader(f)

    for row in reader:

        conn.execute(

            'INSERT INTO Member (id, firstname, lastName) VALUES (?, ?, ?);',

            (int(row['id']), row['firstname'], row['lastName'])

        )

 

conn.commit()

print('Member rows loaded:', conn.execute('SELECT COUNT(*) FROM Member;').fetchone()[0])

Member rows loaded: 4


In [7]:
with open(LOAN_PATH, newline='', encoding='utf-8') as f:

    reader = csv.DictReader(f)

    for row in reader:

        # Convert empty dateReturned to None (→ NULL in SQLite)

        date_returned = row['dateReturned'] if row['dateReturned'].strip() else None

 

        conn.execute(

            '''INSERT INTO Loan (callNo, id, dateBorrowed, dateReturned, dateDue)

               VALUES (?, ?, ?, ?, ?);''',

            (row['callNo'], int(row['id']),

             row['dateBorrowed'], date_returned, row['dateDue'])

        )

 

conn.commit()

print('Loan rows loaded:', conn.execute('SELECT COUNT(*) FROM Loan;').fetchone()[0])

Loan rows loaded: 4


In [8]:
# Retrieve all books ordered by author name
conn = conn = sqlite3.connect(DB_PATH)
books = conn.execute("SELECT * FROM Book ORDER BY author ASC").fetchall()

# Print results
print(f"Total books: {len(books)}")  
for book in books:
    print(book)



Total books: 11
('R 487 T35 1967', 'Medicine in medieval England.', 'Charles H Talbot')
('QA 76.9 D26H39 1996', 'Data model patterns : conventions of thought', 'David Hay')
('CB 351 M293 1983', 'Atlas of medieval Europe', 'Donald Matthew')
('HQ 1143 P68 1975', 'Medieval women', 'Eileen Power')
('PC 14 V48 1965', 'Medieval miscellany', 'Frederick Whitehead')
('QA 76.73 S67C435 2004', "Joe Celko's Trees and hierarchies in SQL for smarties", 'Joe Celko')
('QA 76.73 S67C46 1997', "Joe Celko's SQL puzzles & answers", 'Joe Celko')
('QA 76.9 D35C45 1999', "Joe Celko's data & databases : concepts in practice", 'Joe Celko')
('R 141 E45 2006', 'Medieval medicine and the plague', 'Lynne Elliott')
('QA 76.9 D26H355 2008', 'Information modeling and relational databases', 'T A Halpin')
('QA 76.76 A65P76 2011', 'Programming Android', 'Zigurd R Mednieks')


In [12]:
with sqlite3.connect(DB_PATH) as conn:
    conn.execute("PRAGMA foreign_keys = ON;")  # ensure FKs are enforced
    
    query = """
    SELECT
        Book.title,
        Member.firstname,
        Member.lastname
    FROM Loan
    JOIN Book ON Loan.callNo = Book.callNo
    JOIN Member ON Loan.id = Member.id
    WHERE Loan.dateReturned IS NULL;
    """
    
    results = conn.execute(query).fetchall()
    
    print(f"Total outstanding loans: {len(results)}\n")
    for row in results:
        print(f"Book: {row[0]}, Borrowed by: {row[1]}")
    

Total outstanding loans: 2

Book: Joe Celko's SQL puzzles & answers, Borrowed by: David
Book: Medieval medicine and the plague, Borrowed by: David


In [22]:
with sqlite3.connect(DB_PATH) as conn:
    conn.execute("PRAGMA foreign_keys = ON;")
    
    
    query = """
    SELECT
        Member.firstname,
        Member.lastname,
        Loan.dateReturned
    FROM Loan
    JOIN Member ON Loan.id = Member.id
    WHERE Loan.callNo = 'R 141 E45 2006'
    ORDER BY Loan.dateBorrowed ASC
    """
    
    results = conn.execute(query).fetchall()
    
    # Print member names 
    print(f"Total loans for this book: {len(results)}\n")
    for firstname, lastname, _ in results:  # ignore dateReturned in print
        print(f"Borrowed by: {firstname} {lastname}")
    
    # Returned or Didnt return
    returns = []
    for firstname, lastname, dateReturned in results:
        full_name = f"{firstname} {lastname}"
        if dateReturned:
            returns.append(f"{full_name} returned the book on time")
        else:
            returns.append(f"{full_name} has not yet returned it")
    print(f"\n{'; '.join(returns)}.")

Total loans for this book: 2

Borrowed by: Betty Freeman
Borrowed by: David Martin

Betty Freeman returned the book on time; David Martin has not yet returned it.


In [55]:
with sqlite3.connect(DB_PATH) as conn:
    # Ensure foreign keys are enforced
    conn.execute("PRAGMA foreign_keys = ON;")
    
    query = '''
    SELECT Member.id, Member.firstname, Member.lastName
    FROM Member
    LEFT JOIN Loan ON Member.id = Loan.id
    WHERE Loan.id IS NULL;
    '''

    result = conn.execute(query4).fetchall()


    for row in result:
        print(f" John Martin (id {row[0]}) has no loan record at all")
        print("John Smith (id 1) returned his book so he still appears in Loan")

 John Martin (id 4) has no loan record at all
John Smith (id 1) returned his book so he still appears in Loan


In [34]:
with sqlite3.connect(DB_PATH) as conn:
    conn.execute("PRAGMA foreign_keys = ON;")
    
    query = """
    SELECT
        Member.firstname,
        Member.lastname,
        COUNT(Loan.id) AS total_loans
    FROM Member
    LEFT JOIN Loan ON Member.id = Loan.id
    GROUP BY Member.id
    ORDER BY total_loans DESC
    """
    
    results = conn.execute(query).fetchall()
    
    
    for firstname, lastname, total_loans in results:
        print(f"{firstname} {lastname} — Total loans: {total_loans}")

David Martin — Total loans: 2
John Smith — Total loans: 1
Betty Freeman — Total loans: 1
John Martin — Total loans: 0


In [35]:
with sqlite3.connect(DB_PATH) as conn:
    conn.execute("PRAGMA foreign_keys = ON;")
    
    query = """
    SELECT
        author,
        COUNT(callNo) AS num_books
    FROM Book
    GROUP BY author
    HAVING num_books >= 1
    ORDER BY num_books DESC
    """
    
    results = conn.execute(query).fetchall()
    
    for author, num_books in results:
        if num_books == 1:
            print(f"Author '{author}' has only 1 book in the library.")
        else:
            print(f"Author '{author}' has {num_books} books in the library.")

Author 'Joe Celko' has 3 books in the library.
Author 'Zigurd R Mednieks' has only 1 book in the library.
Author 'T A Halpin' has only 1 book in the library.
Author 'Lynne Elliott' has only 1 book in the library.
Author 'Frederick Whitehead' has only 1 book in the library.
Author 'Eileen Power' has only 1 book in the library.
Author 'Donald Matthew' has only 1 book in the library.
Author 'David Hay' has only 1 book in the library.
Author 'Charles H Talbot' has only 1 book in the library.
